# Map, apply, and vectorize · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 14. Map, apply, and vectorize

Choose an appropriate way to transform values and create categories.

Before you start: You can create columns, filter values, and define a short Python function.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#function-application

### M14C01. Map codes to meaningful labels

`Series.map()` can look up each value in a dictionary. A code missing from the dictionary produces a missing result. Keep the source codes so you can investigate labels that were not mapped.

**Read:** What do the keys of labels represent?

Answer: The codes we want to look up.

**Predict:** Which Label entries will be missing?

Answer: The entry for CS, which is not mapped, and the entry whose source code is missing.

**Explain:** Why should the unmapped CS and missing source code be investigated separately?

Answer: One is an observed but unrecognized code; the other was never recorded.



In [ ]:
# Predict before running: Which Label entries will be missing?
codes = pd.Series(["AI", "IT", "CS", None], dtype="string")
labels = {"AI": "Artificial Intelligence", "IT": "Information Technology"}
result = pd.DataFrame({"Code": codes, "Label": codes.map(labels)})
print(result)

**Your turn:** For this exercise, define CS as Cybersecurity. Add it to the mapping and rebuild the result while leaving the original missing code missing.

For the check, keep these result names: `result`.

**Hint:** Add one dictionary key-value pair; do not fill every missing label.

**Watch for this:** The meaning of a code must come from a stated codebook, not a guess.

In [ ]:
# Change this code to complete the task.
# Question: For this exercise, define CS as Cybersecurity. Add it to the mapping and rebuild the result while leaving the original missing code missing.
codes = pd.Series(["AI", "IT", "CS", None], dtype="string")
labels = {"AI": "Artificial Intelligence", "IT": "Information Technology", "CS": "Cybersecurity"}
result = pd.DataFrame({"Code": codes, "Label": codes.map(labels)})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Label"].isna().sum() == 1
assert result.loc[2, "Label"] == "Cybersecurity"
print("Code checks passed. Review the written explanation as well.")

The CS row now has its specified label; the row without a source code remains missing.

**Quick check:** What happens to an unmapped key in Series.map with a plain dictionary?

1. It is always copied unchanged
2. It produces a missing result
3. It becomes zero

Answer: 2. Unlike a general text replacement, a dictionary map returns missing for keys it does not contain.

**Explain what you learned:** Check what a lookup leaves unmatched.

### M14C02. Apply a function to each value

A Python function can express a custom rule for one value. `Series.apply()` calls that function for each entry. When the same rule can be written with built-in column operations, those operations are often simpler and more efficient.

**Read:** What does the function receive each time it is called?

Answer: One score value, not the whole Series.

**Predict:** Which label does a score of exactly 80 receive?

Answer: Secure, because the condition uses >=.

**Explain:** Is apply required for every calculation on a column?

Answer: No. Built-in arithmetic, comparisons, string methods, and aggregations often do the job directly.



In [ ]:
# Predict before running: Which label does a score of exactly 80 receive?
scores = pd.Series([58, 79, 80, 95])
def status(score):
    return "Secure" if score >= 80 else "Review"
result = scores.apply(status)
print(result.tolist())

**Your turn:** Create the same labels without a custom function. Use a Boolean comparison and a mapping from True/False to labels.

For the check, keep these result names: `result`.

**Hint:** Start with `scores.ge(80)`.

**Watch for this:** These labels use an exercise rule, not an official grading policy.

In [ ]:
# Change this code to complete the task.
# Question: Create the same labels without a custom function. Use a Boolean comparison and a mapping from True/False to labels.
scores = pd.Series([58, 79, 80, 95])
result = scores.ge(80).map({True: "Secure", False: "Review"})
print(result.tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == ["Review", "Review", "Secure", "Secure"]
print("Code checks passed. Review the written explanation as well.")

The labels are unchanged. The column comparison makes the threshold visible without a separate function.

**Quick check:** What does ge(80) mean?

1. Greater than or equal to 80
2. Exactly 80
3. Less than 80

Answer: 1. ge is the named form of the >= comparison.

**Explain what you learned:** Use a custom function when it adds clarity, not by default.

### M14C03. A row function versus column arithmetic

`DataFrame.apply(..., axis=1)` passes one row at a time to a function. Here Quiz is out of 20 and Project is out of 50, so both must be scaled before weighting them. A direct column formula can express the same calculation.

**Read:** Why is axis=1 used here?

Answer: The function needs Quiz and Project from the same row.

**Predict:** What is the first weighted percentage?

Answer: 80.0: 16/20 contributes 32 and 40/50 contributes 48.

**Explain:** Why would Quiz + Project be a different measure?

Answer: It would sum raw points without the stated 40% and 60% weights.



In [ ]:
# Predict before running: What is the first weighted percentage?
df = pd.DataFrame({"Quiz": [16, 12, 18], "Project": [40, 35, 45]})
result = df.apply(lambda row: (row["Quiz"] / 20) * 40 + (row["Project"] / 50) * 60, axis=1)
print(result.round(2).tolist())

**Your turn:** Compute the same weighted percentage with direct column arithmetic, without apply or lambda.

For the check, keep these result names: `df`, `result`.

**Hint:** Perform each calculation on the entire df["Quiz"] or df["Project"] Series.

**Watch for this:** Check the maximum points and weights before calculating a percentage.

In [ ]:
# Change this code to complete the task.
# Question: Compute the same weighted percentage with direct column arithmetic, without apply or lambda.
df = pd.DataFrame({"Quiz": [16, 12, 18], "Project": [40, 35, 45]})
result = (df["Quiz"] / 20) * 40 + (df["Project"] / 50) * 60
print(result.round(2).tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert np.allclose(result.to_numpy(), [80.0, 66.0, 90.0])
assert result.index.equals(df.index)
print("Code checks passed. Review the written explanation as well.")

The percentages remain 80.0, 66.0, and 90.0. Direct arithmetic also preserves the row index.

**Quick check:** With DataFrame.apply, axis=1 passes which object to the function?

1. A row
2. A file
3. The column names only

Answer: 1. axis=1 applies the function across the columns of each row.

**Explain what you learned:** Prefer direct column formulas for ordinary arithmetic.

### M14C04. Fixed intervals versus quantile groups

`cut()` groups numbers using value boundaries that you choose. `qcut()` uses sample quantiles to form groups with roughly similar counts. They answer different questions, and equal-count groups need not have equal numeric widths.

**Read:** Which method uses the explicit boundary of 50?

Answer: cut().

**Predict:** How many values are in the lower fixed band and lower quantile group?

Answer: Three values are in Lower; two are in Lower_half.

**Explain:** Why would quantile bands be unsuitable for an official fixed pass threshold?

Answer: Their boundaries depend on the current sample rather than a fixed scoring rule.



In [ ]:
# Predict before running: How many values are in the lower fixed band and lower quantile group?
values = pd.Series([10, 20, 30, 100])
fixed = pd.cut(values, bins=[0, 50, 100], labels=["Lower", "Upper"], include_lowest=True)
quantile = pd.qcut(values, q=2, labels=["Lower_half", "Upper_half"])
result = pd.DataFrame({"Value": values, "Fixed_band": fixed, "Quantile_band": quantile})
print(result)

**Your turn:** Use students(). Create Score_band using boundaries 0, 60, 80, 100 with labels Developing, Progressing, Secure. Include both 0 and 100, with 60 in Developing and 80 in Progressing. Store the count of each band in a Series named result.

For the check, keep these result names: `result`.

**Hint:** Use cut with right=True and include_lowest=True, then value_counts(sort=False).

**Watch for this:** Tied values can prevent qcut from making the requested number of distinct bins; inspect boundaries and group counts.

In [ ]:
# Change this code to complete the task.
# Question: Use students(). Create Score_band using boundaries 0, 60, 80, 100 with labels Developing, Progressing, Secure. Include both 0 and 100, with 60 in Developing and 80 in Progressing. Store the count of each band in a Series named result.
df = students()
df["Score_band"] = pd.cut(df["Score"], bins=[0, 60, 80, 100], labels=["Developing", "Progressing", "Secure"], right=True, include_lowest=True)
result = df["Score_band"].value_counts(sort=False)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_dict() == {"Developing": 1, "Progressing": 5, "Secure": 6}
print("Code checks passed. Review the written explanation as well.")

The counts are 1 Developing, 5 Progressing, and 6 Secure. These are practice labels with explicit boundary rules.

**Quick check:** Which method is appropriate for fixed score boundaries?

1. qcut
2. cut
3. sort_index

Answer: 2. cut accepts the exact value boundaries you need.

**Explain what you learned:** Choose bands by meaning or by sample distribution, as required.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates